# Savora — Complete Standalone Jupyter Notebook

**All Python source is embedded in this one notebook.** No `savora_core.py` or `requirements.txt` required. Place `RAW_recipes.csv` in the data folder you configure below. Qwen is optional; a rule-based fallback works offline.

## 1. Install dependencies
Run once. Restart the kernel after installing if needed.

In [ ]:
%pip install pandas numpy gradio rapidfuzz ipywidgets transformers torch accelerate sentencepiece

## 2. Complete Savora implementation
This cell contains dataset loading, entity normalization, Qwen NLU, dialogue management, recipe retrieval, and Gradio interface.

In [1]:
"""Savora: notebook-friendly Food.com conversational recipe assistant.
Qwen is optional; rule-based fallback works without a GPU or internet.
"""
import ast
import copy
import json
import re
from dataclasses import dataclass, field
from pathlib import Path

import pandas as pd

ALIASES = {'potatoes':'potato','tomatoes':'tomato','onions':'onion','chillies':'chili',
           'chilies':'chili','capsicum':'bell pepper','bell peppers':'bell pepper',
           'brinjal':'eggplant','aubergine':'eggplant','garbanzo beans':'chickpeas',
           'chickpea':'chickpeas','coriander leaves':'cilantro','curd':'yogurt'}
DIET_ALIASES = {'gluten free':'gluten-free','dairy free':'dairy-free',
                'low carb':'low-carb','low fat':'low-fat','low sodium':'low-sodium',
                'plant based':'vegan'}
CUISINES = {'indian','italian','mexican','chinese','thai','japanese','french',
            'greek','korean','mediterranean','american','vietnamese'}
DIETS = {'vegetarian','vegan','gluten-free','dairy-free','low-carb','low-fat','low-sodium'}


def normalize(text):
    text = re.sub(r'\s+', ' ', str(text).lower().replace('_', ' ').strip())
    text = re.sub(r'[^\w\s-]', '', text)
    return ALIASES.get(text, DIET_ALIASES.get(text, text))


def parse_list(value):
    if isinstance(value, list):
        return value
    if not isinstance(value, str):
        return []
    try:
        out = ast.literal_eval(value)
        return out if isinstance(out, list) else []
    except (SyntaxError, ValueError, TypeError, MemoryError):
        return []


@dataclass
class Slots:
    ingredients: list = field(default_factory=list)
    excluded: list = field(default_factory=list)
    cuisine: str | None = None
    diet: str | None = None
    max_minutes: int | None = None


class RecipeStore:
    def __init__(self, path, max_rows=None):
        path = Path(path)
        if not path.exists():
            raise FileNotFoundError(f'{path} not found. Set DATA_DIR to your CSV folder.')
        cols = ['id','name','minutes','ingredients','tags','steps','description']
        self.df = pd.read_csv(path, usecols=cols, nrows=max_rows, low_memory=False)
        self.df['minutes'] = pd.to_numeric(self.df['minutes'], errors='coerce')
        self.df['name'] = self.df['name'].fillna('Untitled recipe')
        for col in ('ingredients','tags','steps'):
            self.df[col] = self.df[col].map(parse_list)
        self.df['ingredient_set'] = self.df['ingredients'].map(
            lambda values: {normalize(v) for v in values})
        self.df['tag_set'] = self.df['tags'].map(
            lambda values: {normalize(v) for v in values})
        self.vocabulary = set().union(*self.df['ingredient_set'].tolist()) if len(self.df) else set()
        self.vocabulary.update(ALIASES)

    def search(self, slots, limit=5):
        df = self.df
        if slots.max_minutes is not None:
            df = df[df['minutes'].le(slots.max_minutes)]
        if slots.cuisine:
            df = df[df['tag_set'].map(lambda x: slots.cuisine in x)]
        if slots.diet:
            df = df[df['tag_set'].map(lambda x: slots.diet in x)]
        if slots.ingredients:
            df = df[df['ingredient_set'].map(lambda x: all(v in x for v in slots.ingredients))]
        if slots.excluded:
            df = df[df['ingredient_set'].map(lambda x: not any(v in x for v in slots.excluded))]
        return df.sort_values(['minutes','id']).head(limit).to_dict('records')


class NLU:
    def __init__(self, vocabulary, use_qwen=False, model_id='Qwen/Qwen2.5-1.5B-Instruct'):
        self.vocabulary = vocabulary
        self.pipe = None
        if use_qwen:
            try:
                import torch
                from transformers import pipeline
                kwargs = {'model': model_id, 'device_map': 'auto', 'torch_dtype': 'auto'}
                self.pipe = pipeline('text-generation', **kwargs)
            except Exception as exc:
                raise RuntimeError('Could not load Qwen. Install torch, transformers and accelerate; '
                                   'or use use_qwen=False. Original error: ' + str(exc)) from exc

    def _rules(self, utterance):
        text = normalize(utterance)
        if re.search(r'\b(reset|start over|clear preferences)\b', text):
            return {'intent':'reset'}
        if re.search(r'\b(help|what can you do)\b', text):
            return {'intent':'help'}
        cuisine = next((x for x in sorted(CUISINES, key=len, reverse=True)
                        if re.search(r'(?<!\w)'+re.escape(x)+r'(?!\w)', text)), None)
        diet = next((x for x in sorted(DIETS, key=len, reverse=True)
                     if re.search(r'(?<!\w)'+re.escape(x)+r'(?!\w)', text)), None)
        for alias, value in DIET_ALIASES.items():
            if re.search(r'(?<!\w)'+re.escape(alias)+r'(?!\w)', text):
                diet = value
        m = re.search(r'\b(?:under|less than|within|max(?:imum)?|up to|in)\s*(\d+)\s*(?:minutes?|mins?)\b', text)
        minutes = int(m.group(1)) if m else None
        positive, excluded = [], []
        for term in sorted(self.vocabulary, key=len, reverse=True):
            if len(term) < 3 or not re.search(r'(?<!\w)'+re.escape(term)+r'(?!\w)', text):
                continue
            canonical = normalize(term)
            if re.search(r'\b(?:without|exclude|remove|no|avoid)\s+(?:any\s+)?'+re.escape(term)+r'(?!\w)', text):
                if canonical not in excluded:
                    excluded.append(canonical)
            elif canonical not in positive:
                positive.append(canonical)
        # Prefer longer matched ingredient names over their embedded substrings.
        positive = [x for x in positive if not any(x != y and x in y for y in positive)]
        return {'intent':'search','ingredients':positive[:10], 'excluded':excluded[:10],
                'cuisine':cuisine,'diet':diet,'max_minutes':minutes}

    def extract(self, utterance):
        fallback = self._rules(utterance)
        if self.pipe is None or fallback.get('intent') in ('reset','help'):
            return fallback
        instruction = ('Extract recipe preferences from the user message. Return ONLY JSON with keys '
                       'intent, ingredients, excluded, cuisine, diet, max_minutes. '
                       'Use null for absent scalar values and [] for absent arrays. '
                       'Intent is search, reset or help. Do not invent preferences.')
        try:
            messages = [{'role':'system','content':instruction},
                        {'role':'user','content':utterance}]
            result = self.pipe(messages, max_new_tokens=180, do_sample=False)
            generated = result[0]['generated_text']
            answer = generated[-1]['content'] if isinstance(generated, list) else generated
            match = re.search(r'\{.*?\}', answer, flags=re.S)
            obj = json.loads(match.group()) if match else {}
            clean = {'intent':obj.get('intent','search')}
            for key in ('ingredients','excluded'):
                values = obj.get(key) or []
                if isinstance(values, str): values = [values]
                clean[key] = list(dict.fromkeys(normalize(x) for x in values
                                if isinstance(x, str) and normalize(x) in self.vocabulary))[:10]
            for key, allowed in (('cuisine',CUISINES),('diet',DIETS)):
                value = normalize(obj[key]) if isinstance(obj.get(key),str) else None
                clean[key] = value if value in allowed else None
            try:
                val = obj.get('max_minutes')
                clean['max_minutes'] = int(val) if val is not None and 0 < int(val) < 10000 else None
            except (ValueError,TypeError):
                clean['max_minutes'] = None
            # If model fails to recognize a clear constraint, use the rule-based result.
            for key in ('ingredients','excluded','cuisine','diet','max_minutes'):
                if not clean.get(key): clean[key] = fallback.get(key)
            return clean
        except (ValueError,TypeError,KeyError,IndexError,AttributeError):
            return fallback


class SavoraBot:
    def __init__(self, data_dir='.', use_qwen=False,
                 model_id='Qwen/Qwen2.5-1.5B-Instruct', max_rows=None):
        self.store = RecipeStore(Path(data_dir)/'RAW_recipes.csv', max_rows=max_rows)
        self.nlu = NLU(self.store.vocabulary, use_qwen=use_qwen, model_id=model_id)
        self.state = Slots()
        self.last_results = []

    def reset(self):
        self.state = Slots()
        self.last_results = []

    def respond(self, message):
        if not str(message).strip(): return 'Please describe what you would like to cook.'
        obj = self.nlu.extract(message)
        if obj.get('intent') == 'reset':
            self.reset()
            return 'Preferences cleared. What would you like to cook?'
        if obj.get('intent') == 'help':
            return ('Try: "Vegetarian Indian recipe with potato under 30 minutes". '
                    'Follow up with "under 20 minutes", "without onion", or "reset".')
        for key in ('cuisine','diet','max_minutes'):
            if obj.get(key) is not None: setattr(self.state,key,obj[key])
        for item in obj.get('excluded',[]):
            if item not in self.state.excluded: self.state.excluded.append(item)
            if item in self.state.ingredients: self.state.ingredients.remove(item)
        for item in obj.get('ingredients',[]):
            if item not in self.state.ingredients: self.state.ingredients.append(item)
            if item in self.state.excluded: self.state.excluded.remove(item)
        if not any((self.state.ingredients,self.state.excluded,self.state.cuisine,
                    self.state.diet,self.state.max_minutes)):
            return 'What ingredients, cuisine, diet, or cooking time would you prefer?'
        self.last_results = self.store.search(self.state)
        if not self.last_results:
            return ('No exact matches for these preferences. Try a longer cooking time, '
                    'fewer required ingredients, or say "reset".')
        lines = ['Here are recipes matching your current preferences:']
        for index, rec in enumerate(self.last_results,1):
            mins = f"{rec['minutes']:g}" if pd.notna(rec['minutes']) else '?'
            lines.append(f"{index}. **{str(rec['name']).title()}** — {mins} min (recipe ID {rec['id']})")
            lines.append('   Ingredients: '+', '.join(map(str,rec['ingredients'][:8])))
        lines.append('Refine your search with a follow-up, or type "reset".')
        return '\n'.join(lines)

    def recipe_details(self, result_number=1):
        if not 1 <= result_number <= len(self.last_results):
            return 'Choose a recipe number from your most recent results.'
        rec = self.last_results[result_number-1]
        steps = '\n'.join(f'{i}. {step}' for i,step in enumerate(rec['steps'],1))
        return (f"{rec['name'].title()}\nIngredients: {', '.join(rec['ingredients'])}\n"
                f"Instructions:\n{steps}")


def launch_gradio(bot, share=False):
    import gradio as gr
    # Separate state per browser conversation rather than sharing one global bot.
    def chat(message, history, state):
        if state is None: state = Slots()
        session_bot = copy.copy(bot)
        session_bot.state = state
        response = session_bot.respond(message)
        return response, session_bot.state
    with gr.Blocks(title='Savora Recipe Assistant') as demo:
        gr.Markdown('# Savora — Conversational Recipe Assistant')
        gr.Markdown('Ask for recipes by ingredients, cuisine, diet, or cooking time.')
        state = gr.State(None)
        chatbox = gr.Chatbot(type='messages')
        user_input = gr.Textbox(label='Your message', placeholder='Vegetarian Indian potato dish under 30 minutes')
        def submit(message, history, session_state):
            reply, session_state = chat(message, history, session_state)
            history = history + [{'role':'user','content':message},{'role':'assistant','content':reply}]
            return '', history, session_state
        user_input.submit(submit,[user_input,chatbox,state],[user_input,chatbox,state])
        gr.Button('Reset chat').click(lambda: ('',[],None),None,[user_input,chatbox,state])
    return demo.launch(share=share, inbrowser=False)


## 1. Configure dataset path

Change `DATA_DIR` to the folder containing `RAW_recipes.csv`. For quicker CPU tests, set `MAX_ROWS=20000`; set it to `None` for the entire dataset.

In [4]:
from pathlib import Path

DATA_DIR = Path(r"C:\Users\Aryaman\Downloads\Food.com\Food.com")

USE_QWEN = False
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
MAX_ROWS = 10000

print("Dataset exists:", (DATA_DIR / "RAW_recipes.csv").exists())

Dataset exists: True


In [3]:
from pathlib import Path

search_folders = [
    Path.cwd(),
    Path.home() / "Downloads",
    Path.home() / "Documents",
    Path.home() / "Desktop"
]

matches = []

for folder in search_folders:
    if folder.exists():
        matches.extend(folder.rglob("RAW_recipes.csv"))

if matches:
    CSV_PATH = matches[0]
    DATA_DIR = CSV_PATH.parent

    print("Dataset found!")
    print("Location:", CSV_PATH)
else:
    print("Dataset not found. Check where you saved the CSV.")

USE_QWEN = False
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
MAX_ROWS = 10000

Dataset found!
Location: c:\Users\Aryaman\Downloads\Food.com\Food.com\RAW_recipes.csv


In [5]:
bot = SavoraBot(data_dir=DATA_DIR, use_qwen=USE_QWEN, model_id=MODEL_ID, max_rows=MAX_ROWS)
print("Recipes loaded:", len(bot.store.df))
print("Ingredient vocabulary:", len(bot.store.vocabulary))

Recipes loaded: 10000
Ingredient vocabulary: 5337


## 3. Entity normalization

In [6]:
for ingredient in ["potatoes", "brinjal", "capsicum", "garbanzo beans", "gluten free"]:
    print(f"{ingredient:20} -> {normalize(ingredient)}")

potatoes             -> potato
brinjal              -> eggplant
capsicum             -> bell pepper
garbanzo beans       -> chickpeas
gluten free          -> gluten-free


## 4. Test NLU extraction

In [7]:
utterance = "I want a vegetarian Indian dish with potatoes under 30 minutes without onions"
print(bot.nlu.extract(utterance))

{'intent': 'search', 'ingredients': ['potato'], 'excluded': ['onion'], 'cuisine': 'indian', 'diet': 'vegetarian', 'max_minutes': 30}


## 5. Dialogue management and recipe search

In [8]:
print(bot.respond("vegetarian Indian dish with potato under 30 minutes"))
print("\n--- FOLLOW-UP ---\n")
print(bot.respond("make it under 20 minutes"))
print("\nCurrent preferences:", bot.state)

Here are recipes matching your current preferences:
1. **Tide Me Over   Indian Chaat  Simple Veggie Salad** — 2 min (recipe ID 63793)
   Ingredients: cucumber, potato, chopped tomato, plain yogurt, chat masala, paprika, salt
2. **Aloo  Potato  Pakora** — 10 min (recipe ID 110215)
   Ingredients: potatoes, corn oil, besan, water, red chili powder, cumin powder, garlic, ginger
3. **Aloo Curry  Potato Curry** — 25 min (recipe ID 11512)
   Ingredients: potatoes, onions, garlic, ginger, tomatoes, green chilies, salt, turmeric
4. **Aaloo Chaat  Tangy Potatoes** — 25 min (recipe ID 96178)
   Ingredients: potato, red chili powder, chat masala, coriander powder, onion, salt, tamarind pulp, oil
5. **Aloo Baigain  Potato And Eggplant  Aubergine** — 25 min (recipe ID 149969)
   Ingredients: potato, eggplants, red chili powder, cumin, coriander powder, turmeric powder, salt, garlic paste
Refine your search with a follow-up, or type "reset".

--- FOLLOW-UP ---

Here are recipes matching your current

## 6. Inspect a returned recipe

In [9]:
print(bot.recipe_details(1))

Tide Me Over   Indian Chaat  Simple Veggie Salad
Ingredients: cucumber, potato, chopped tomato, plain yogurt, chat masala, paprika, salt
Instructions:
1. toss everything well in a serving bowl
2. grab a fork or large spoon and eat immediately , or refrigerate until hunger strikes , and then eat !
3. feel extremely pleased with yourself that you snacked on this instead of that bag of chips you were eyeing
4. proceed to eat several helpings of dessert after dinner as reward


## 7. Reset preferences

In [10]:
print(bot.respond("reset"))

Preferences cleared. What would you like to cook?


## 8. Optional: enable Qwen NLU

Set `USE_QWEN=True` and run this cell only when your computer has sufficient RAM/VRAM. The first run downloads model weights from Hugging Face. No model training is performed.

In [ ]:
#qwen_bot = SavoraBot(data_dir=DATA_DIR, use_qwen=True, model_id=MODEL_ID, max_rows=MAX_ROWS)
#print(qwen_bot.nlu.extract("I need a vegan Mexican dish in 25 minutes"))

## 9. Interactive notebook chat (ipywidgets)

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

chat_input = widgets.Text(placeholder="Ask Savora for a recipe...", description="You:", layout=widgets.Layout(width="85%"))
send = widgets.Button(description="Send", button_style="success")
reset = widgets.Button(description="Reset")
output = widgets.Output()

def on_send(_):
    msg = chat_input.value.strip()
    if not msg: return
    chat_input.value = ""
    with output:
        display(Markdown("**You:** " + msg.replace("<", "&lt;")))
        display(Markdown("**Savora:**\n\n" + bot.respond(msg)))

def on_reset(_):
    bot.reset()
    with output:
        clear_output()
        print("Conversation reset.")

send.on_click(on_send)
reset.on_click(on_reset)
display(widgets.HBox([chat_input, send, reset]), output)

Output()

In [ ]:
# Uncomment to launch:
# from savora_core import launch_gradio
# launch_gradio(bot)